In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio


colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files_A = sorted((project_root/'data/260519').glob('A_SUBT_0[1-4]*.csv'))
files_B = sorted((project_root/'data/260519').glob('B_SUBT_0[1-4]*.csv'))
acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
#print(files_A)
#print(files_B)

In [3]:

file_names = ["MATCHED_SUBT_01_NewSourceTest_Background", "MATCHED_SUBT_02_NewSourceTest_Cs137",
              "MATCHED_SUBT_03_NewSourceTest_Na22", "MATCHED_SUBT_04_NewSourceTest_Co60"]

#ONLY RUN THIS ONCE!
for i, (file_B, file_A) in enumerate(zip(files_B, files_A)):
    match_events(file_B, file_A, output_path=f"../data/260519/{file_names[i]}.csv", tolerance = 500)

files_matched = sorted((project_root/'data/260519').glob('MATCHED_SUBT_0[1-4]*.csv'))
acqs_matched = [MatchedSiphraAcquisition(file) for file in files_matched]
print(acqs_matched)


========== MATCH RESULTS ==========
Events in file A:     131192
Events in file B:     367781
Matched events:       79244
Discarded from A:     51948
Discarded from B:     288537

Match fraction A: 60.40%
Match fraction B: 21.55%
Saved 79244 matched events
Output file: ../data/260519/MATCHED_SUBT_01_NewSourceTest_Background.csv

========== MATCH RESULTS ==========
Events in file A:     197667
Events in file B:     302139
Matched events:       115907
Discarded from A:     81760
Discarded from B:     186232

Match fraction A: 58.64%
Match fraction B: 38.36%
Saved 115907 matched events
Output file: ../data/260519/MATCHED_SUBT_02_NewSourceTest_Cs137.csv

========== MATCH RESULTS ==========
Events in file A:     157665
Events in file B:     341701
Matched events:       103421
Discarded from A:     54244
Discarded from B:     238280

Match fraction A: 65.60%
Match fraction B: 30.27%
Saved 103421 matched events
Output file: ../data/260519/MATCHED_SUBT_03_NewSourceTest_Na22.csv

========== MA

In [4]:
# histograms
hists = {}

acqs = [acqs_A, acqs_B, acqs_matched]


siphras = ['A', 'B', 'Summed']

for i, (siphra, acq), in enumerate(zip(siphras, acqs)):
    bg = acq[0]
    sources = []

    for sgnl in acq[1:]:
        filepath = sgnl.filepath.stem
        src = (MetadataLoader.load(sgnl.metadataFile)).source
        print(src)
        sources.append(src)
        # Signal and Background
        hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
        hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
        if siphra == 'Summed':
            hist_sgnl.Fill(sgnl['Combined_Summed']/(len(sgnl.active_chs)*2))
            hist_bg.Fill(bg['Combined_Summed']/(len(bg.active_chs)*2))
        else:
            hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
            hist_bg.Fill(bg['s']/len(bg.active_chs))
        hist_bg.Scale(sgnl.exposure/bg.exposure)
        # Clean spectrum
        hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
        hist_clean.Add(hist_bg, -1)
        for hist in [hist_sgnl, hist_bg, hist_clean]:
            # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")
        hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
        hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
        hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
        del hist_sgnl, hist_bg

Cs-137
Na-22
Co-60
Cs-137
Na-22
Co-60
Cs-137
Na-22
Co-60


In [5]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 3200, 2400)
cv.Divide(3,3)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(9)]

for i, siphra in enumerate(siphras):
    for idx, src in enumerate(sources):   
        cv.cd(3*i + idx+1)
        lg = lgds[3*i + idx]
        cv.cd(3*i + idx+1).SetLogy()

        print(src)
        print(f"{i}, {idx}")
        
        sgnl = hists[f"{src}_SIPHRA_{siphra}"] 
        bg = hists[f"{src}_BG_SIPHRA_{siphra}"] 
        clean = hists[f"{src}_clean_SIPHRA_{siphra}"]
        lg.AddEntry(sgnl, "Signal")
        lg.AddEntry(bg, "Background")
        lg.AddEntry(clean, "Subtracted")
        sgnl.SetLineColor(colors[0])
        bg.SetLineColor(colors[1])
        clean.SetLineColor(colors[2])
        if siphra == "Summed":
            sgnl.SetTitle(f"{src}, Summed detector A and B")
        else:
            sgnl.SetTitle(f"{src}, Detector setup {siphra}")
        sgnl.Draw("hist")
        bg.Draw("sames hist")
        clean.Draw("sames hist")
        ROOT.gPad.Update()
        for j, sp in enumerate([sgnl, bg, clean]):
            st = sp.FindObject("stats")
            # print(type(st))
            st.SetY1NDC(Yinit - j*0.08)
            st.SetY2NDC(0.1)
        lg.Draw()
        
        cv.Draw()

Cs-137
0, 0
Na-22
0, 1
Co-60
0, 2
Cs-137
1, 0
Na-22
1, 1
Co-60
1, 2
Cs-137
2, 0
Na-22
2, 1
Co-60
2, 2


In [6]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum
siphra = "Summed"
src = "Na-22"

hist = hists[f"{src}_clean_SIPHRA_{siphra}"]
energy_ranges = [(80,150), (280, 340), (390, 440)]
energies = NA22_MEV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.004272800649888127, -0.023576474258143597]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      64.2437
NDf                       =           67
Edm                       =  6.24749e-06
NCalls                    =           65
Constant                  =      154.116   +/-   3.3684      
Mean                      =      122.524   +/-   0.648407    
Sigma                     =      22.7666   +/-   0.681004     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      45.8873
NDf                       =           57
Edm                       =  5.29308e-07
NCalls                    =           79
Constant                  =      579.485   +/-   5.98521     
Mean                      =        310.9   +/-   0.235316    
Sigma                     =      20.4156   +/-   0.304912     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2             

In [7]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][1], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
#hist_cal_bg_Cs137.SetTitle(r" ^{137}Cs, 2-sided setup detector A")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.8)])
print(res_Cs137)



Mean: 0.6164447131596819
Mean error: 0.0005539347116697422
Sigma: 0.11845808441272769
Sigma error: 0.0006866879709986479
Covariance: -2.1932412486187475e-07
Resolution Error: 0.28709641106904693 %
[0.45158388485976086]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       66.563
NDf                       =           67
Edm                       =  9.70424e-09
NCalls                    =           93
Constant                  =      24.0519   +/-   0.0796512   
Mean                      =     0.616445   +/-   0.000553935 
Sigma                     =     0.118458   +/-   0.000686688  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [8]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{137}Cs 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][1], BITS12, title = r"^{137}Cs 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.85)])
print(res_Cs137)


Mean: 0.6477949127270052
Mean error: 0.0005372003176154432
Sigma: 0.12154976729955595
Sigma error: 0.0006495907595180789
Covariance: -1.4937274749361473e-07
Resolution Error: 0.25346736854217533 %
[0.44094503915058103]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      184.437
NDf                       =           78
Edm                       =  1.21772e-06
NCalls                    =           83
Constant                  =      16.5877   +/-   0.0630952   
Mean                      =     0.647795   +/-   0.0005372   
Sigma                     =      0.12155   +/-   0.000649591  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [9]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][1], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.75)])
print(res_Cs137)


Mean: 0.6236912408574163
Mean error: 0.0004751147487040667
Sigma: 0.09104718217689504
Sigma error: 0.0006637938410106875
Covariance: -2.5412805517261205e-08
Resolution Error: 0.2535575409275763 %
[0.34305576878322314]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      66.2671
NDf                       =           55
Edm                       =  7.55034e-06
NCalls                    =           82
Constant                  =      13.7719   +/-   0.0685493   
Mean                      =     0.623691   +/-   0.000475115 
Sigma                     =    0.0910472   +/-   0.000663794  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [10]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][2], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, ((0.35, 0.65), (1, 1.45), (1.6, 1.8)))
print(res_Cs137)


Mean: 0.4933802957193986
Mean error: 0.0012529368904241598
Sigma: 0.1174126064810871
Sigma error: 0.0018344270899178852
Covariance: -2.3628461731498213e-07
Resolution Error: 0.899510857903884 %
Mean: 1.2640466711989917
Mean error: 0.001404011367182402
Sigma: 0.1251776657060258
Sigma error: 0.001606670244726001
Covariance: 4.956361223606365e-07
Resolution Error: 0.29410109922180655 %
Mean: 1.6801516216699555
Mean error: 0.007760677390517446
Sigma: 0.13362473397093463
Sigma error: 0.015596738554003266
Covariance: -7.54073773672234e-05
Resolution Error: 2.2362916912712407 %
[0.5592433010082736, 0.23271887115539225, 0.18689868270316218]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      80.9793
NDf                       =           68
Edm                       =  1.99877e-08
NCalls                    =           70
Constant                  =      2.03198   +/-   0.0186343   
Mean                      =      0.49338   +/-   0.00125294  


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [11]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][2], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, ((0.36, 0.65), (1.1, 1.6), (1.7,2.0)))
print(res_Cs137)

Mean: 0.515185326956259
Mean error: 0.0027902790216164756
Sigma: 0.13417578918239917
Sigma error: 0.004878368778535813
Covariance: 3.004209466642128e-06
Resolution Error: 2.17624154787056 %
Mean: 1.3412318116323207
Mean error: 0.0013168318566619771
Sigma: 0.1336147729789516
Sigma error: 0.0014413085179074414
Covariance: -7.674994673932786e-08
Resolution Error: 0.2545024682351596 %
Mean: 1.8321209260874443
Mean error: 0.004061809867898169
Sigma: 0.1297977720301242
Sigma error: 0.005902929077707571
Covariance: -1.1390097516650192e-05
Resolution Error: 0.7753636935068762 %
[0.6120382085443385, 0.234109207504104, 0.16648724433391113]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      50.2266
NDf                       =           65
Edm                       =    1.822e-06
NCalls                    =           89
Constant                  =     0.797001   +/-   0.0126965   
Mean                      =     0.515185   +/-   0.00279028  
Sig

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [12]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][2], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.36, 0.62), (1.15, 1.45), (1.63, 1.9)])
print(res_Cs137)

Mean: 0.500969011447033
Mean error: 0.002579014736063464
Sigma: 0.09309836657682485
Sigma error: 0.003306930119076165
Covariance: 3.3275747016446977e-06
Resolution Error: 1.4780994736127602 %
Mean: 1.3042375709378473
Mean error: 0.0009286576335251225
Sigma: 0.08792477821343427
Sigma error: 0.001077538844669844
Covariance: 1.1469802996456008e-07
Resolution Error: 0.1931853050467793 %
Mean: 1.7665537949601413
Mean error: 0.002000636458071938
Sigma: 0.0886530589581146
Sigma error: 0.0025993984868226128
Covariance: -7.411049853690418e-07
Resolution Error: 0.34794569073063136 %
[0.436715957387456, 0.15842453354030622, 0.11793283009322111]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      52.5904
NDf                       =           58
Edm                       =  2.01636e-06
NCalls                    =           80
Constant                  =     0.436227   +/-   0.0101229   
Mean                      =     0.500969   +/-   0.00257901  

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [13]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][0], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][3], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.1, 2.5)])
print(res_Cs137)

Mean: 2.2802670874160405
Mean error: 0.006771473777848994
Sigma: 0.13380435436750693
Sigma error: 0.008360550406987063
Covariance: -1.2921977616521634e-05
Resolution Error: 0.8718812342626635 %
[0.13789622912987776]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      89.7551
NDf                       =           91
Edm                       =  4.45204e-06
NCalls                    =           73
Constant                  =    0.0856536   +/-   0.00380083  
Mean                      =      2.28027   +/-   0.00677147  
Sigma                     =     0.133804   +/-   0.00836055   	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [14]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][0], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][3], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.25, 2.67)])
print(res_Cs137)

Mean: 2.4389901753788443
Mean error: 0.006090894236579949
Sigma: 0.1316175400088797
Sigma error: 0.0070657909297798804
Covariance: -1.355953410120566e-05
Resolution Error: 0.6914301468271383 %
[0.12681527877529236]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      138.888
NDf                       =           95
Edm                       =  4.08949e-06
NCalls                    =           81
Constant                  =     0.099582   +/-   0.00401038  
Mean                      =      2.43899   +/-   0.00609089  
Sigma                     =     0.131618   +/-   0.00706579   	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [15]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][3], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(2.28, 2.5)])
print(res_Cs137)

Mean: 2.3813999556145093
Mean error: 0.006202468685831252
Sigma: 0.09934975046028605
Sigma error: 0.01064053602083756
Covariance: -1.3907256697090939e-05
Resolution Error: 1.0556994774843518 %
[0.09803977405442836]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      48.3221
NDf                       =           49
Edm                       =  5.04454e-08
NCalls                    =           93
Constant                  =     0.110965   +/-   0.00542291  
Mean                      =       2.3814   +/-   0.00620247  
Sigma                     =    0.0993498   +/-   0.0106405    	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
